# 딥러닝응용III — Lecture 04 실습
# 사전학습 언어모델과 트랜스포머 (교수용 — 답안 포함)

| 항목 | 내용 |
|------|------|
| **강의** | 딥러닝응용III (자연어처리) |
| **수업** | 4주차 — 사전학습 언어모델과 트랜스포머 |
| **교수** | 유원상 교수 |
| **모델** | KcBERT-base (`beomi/kcbert-base`) |

> **교수용** — 모든 문제의 답안과 풀이가 포함되어 있습니다. (🔑 표시)
> 강의 중에는 학생 버전을 학생들에게 배포하고, 교수님은 이 버전을 보면서 한 줄씩 설명합니다.

### 🎯 학습 목표

1. **언어 모델 (Language Model)** — 단어 시퀀스에 확률을 부여하는 모델의 의미와 종류 (순방향/역방향/마스크) 이해
2. **트랜스포머 구조 (Transformer)** — 인코더/디코더 구조, 트랜스포머 블록의 구성 요소 (Multi-Head Attention, FFN, Residual, LayerNorm)
3. **셀프 어텐션 (Self-Attention)** — 쿼리(Query)/키(Key)/밸류(Value)의 역할과 가중합(weighted sum) 계산 원리
4. **BERT 입출력 형식** — `input_ids`, `attention_mask`, `token_type_ids`, `last_hidden_state`, `pooler_output`
5. **임베딩 분석** — `[CLS]` 벡터를 이용한 문장 유사도, 문맥 의존적 단어 임베딩의 의미

### 📋 강의 시 유의 사항

- 모든 빈칸/오류/계산 문제에 정답이 포함되어 있습니다.
- 핵심 개념은 별도의 박스로 강조했습니다.
- Remix와 Demo 활동은 모두 제외하였습니다 (학생들의 강한 반대로 인해).
- 학생들이 코드를 따라가며 자연스럽게 이해하도록 셀 단위로 질문을 배치했습니다.


---
## 1단계. 실습 환경 만들기

> ⚙️ **런타임 설정**: 상단 메뉴 → **런타임** → **런타임 유형 변경** → **GPU** 선택을 권장합니다.
> CPU에서도 동작하지만 임베딩 추출에 수십 초가 소요될 수 있습니다.

### 📦 사용 모델: KcBERT-base

| 항목 | 내용 |
|------|------|
| 모델명 | `beomi/kcbert-base` |
| 학습 데이터 | 네이버 댓글 12GB (한국어 구어체 특화) |
| 구조 | BERT-base: 12 레이어, 12 헤드, hidden 768차원 |
| 학습 방식 | MLM (Masked Language Model) + NSP (Next Sentence Prediction) |
| 파라미터 수 | 약 1억 1천만 개 |

> 💡 **왜 KcBERT인가?**
> 원본 multilingual BERT는 104개 언어를 동시에 학습하므로 한국어 전용 데이터로 학습한
> KcBERT 대비 한국어 성능이 낮습니다. NSMC 같은 구어체 한국어 태스크에서는
> KcBERT가 훨씬 적합합니다.


### ❓ Q1-1. 사전학습(Pre-training) vs 파인튜닝(Fine-tuning)

> (a) `transformers.from_pretrained("beomi/kcbert-base")`가 하는 일은 무엇인가요?
>
> (b) 사전학습된 BERT를 그대로 사용하지 않고 파인튜닝(fine-tuning)을 하는 이유는?
>
> (c) GPU vs CPU — BERT 추론에서 GPU가 유리한 이유 한 가지는?

**🔑 답:**
(a) HuggingFace Hub에서 `beomi/kcbert-base`라는 사전학습 가중치(checkpoint, ~440MB)와 설정(config), 토크나이저 어휘를 다운로드하여 모델 객체를 초기화합니다.
(b) 사전학습은 일반 언어 이해 능력만 학습합니다. 문서 분류·감성 분석 같은 **다운스트림 태스크**는 그 위에 분류기를 얹어 소량의 라벨 데이터로 추가 학습(fine-tuning)해야 합니다.
(c) BERT는 행렬 곱 연산이 매우 많은데, GPU는 수천 개의 코어로 행렬 곱을 **병렬화**하여 CPU 대비 10~100배 빠릅니다.


### 📝 Q1-2. 빈칸 채우기 — 패키지 설치 및 디바이스 설정

> **다음 코드 셀의 `______` 빈칸을 채워서 실행하세요.**
> 학생 버전은 빈칸이 비어 있으므로, 강의노트와 강의노트북을 참고해 직접 채워 넣어야 합니다.


In [1]:
# 🔑 Q1-2 (교수용 정답): 빈칸이 모두 채워진 상태
!pip install -q transformers torch          # 🔑 (a) transformers (b) torch

import torch                                # 🔑 (c) torch
import transformers                         # 🔑 (d) transformers

print(f"PyTorch     버전: {torch.__version__}")
print(f"Transformers 버전: {transformers.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")

# DEVICE: 모델과 텐서를 올릴 디바이스 객체
#   torch.device("cuda") → GPU,  torch.device("cpu") → CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 🔑 (e) torch  (f) cuda  (g) torch  (h) is_available  (i) cpu
print(f"\n사용 디바이스: {DEVICE}")


PyTorch     버전: 2.10.0+cpu
Transformers 버전: 5.0.0
GPU 사용 가능: False

사용 디바이스: cpu


### 🐛 Q1-3. 오류 수정 — 디바이스 설정 (3개)

> 아래 코드에는 **3개의 오류**가 있습니다. 찾아 수정하세요.
>
> ```python
> import torch as transformers          # 오류 1
> DEVICE = torch.cuda("cuda")           # 오류 2
> model.cuda()                           # 모델은 어디에? — 오류 3
> ```

**🔑 답:**
1. `import torch as transformers` → `import torch` (별칭이 잘못 사용됨; transformers와 torch는 별개의 라이브러리)
2. `torch.cuda("cuda")` → `torch.device("cuda")` (디바이스 객체 생성 함수는 `torch.device`)
3. GPU 사용 가능성 확인 없이 `cuda()` 호출 → `model.to(DEVICE)` 또는 `model.to("cuda" if torch.cuda.is_available() else "cpu")`로 안전하게 처리


---
## 2단계. 언어 모델 (Language Model) 개념 직접 체험

### 🔬 언어 모델이란?

언어 모델은 **단어 시퀀스에 확률을 부여하는 모델**입니다. 잘 학습된 한국어 언어 모델은 자연스러운 문장에 더 높은 확률을 부여합니다.

```
P(난폭, 운전) > P(무모, 운전)        ← 더 자연스러운 표현에 높은 확률
P(두시 삼십 이분) > P(이시 서른 두분)
```

### 🧮 수학적 정의

$$P(w_1, w_2, w_3, \ldots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \ldots, w_{i-1})$$

전체 단어 시퀀스가 나타날 결합 확률(joint probability)은 **이전 단어들이 주어졌을 때 다음 단어가 등장할 조건부 확률들의 연쇄(chain rule)**로 분해됩니다.

### 📊 학습 방식 비교

| 방식 | 방향 | 대표 모델 | 핵심 아이디어 |
|------|------|----------|-------------|
| **순방향(Forward)** LM | 왼쪽→오른쪽 | GPT, ELMo | $P(w_i \mid w_1, \ldots, w_{i-1})$ |
| **역방향(Backward)** LM | 오른쪽→왼쪽 | ELMo | $P(w_i \mid w_{i+1}, \ldots, w_n)$ |
| **마스크(Masked)** LM | 양방향 | BERT, KcBERT | $P(w_i \mid w_{1:i-1}, w_{i+1:n})$ |
| **스킵-그램** | 중심→주변 | Word2Vec | 중심 단어로 주변 단어 예측 |


### ❓ Q2-1. 언어 모델 기본 개념

> (a) 결합 확률 $P(w_1, w_2, w_3)$를 조건부 확률로 분해해서 쓰세요.
>
> (b) 확률값이 항상 1보다 작은데, 긴 문장의 확률이 매우 작은 이유는?
>
> (c) "두시 삼십 이분" vs "이시 서른 두분" — 잘 학습된 언어 모델이 어느 쪽에 더 높은 확률을 줄까요? 왜?

**🔑 답:**
(a) $P(w_1, w_2, w_3) = P(w_1) \times P(w_2 \mid w_1) \times P(w_3 \mid w_1, w_2)$
(b) 1보다 작은 확률값들이 **곱해지므로** 길이가 길어질수록 기하급수적으로 작아짐. 이 때문에 실제로는 **로그 확률(log probability)**의 합으로 계산함.
(c) "두시 삼십 이분"이 한국어 시간 표현으로 자연스러우므로 **더 높은 확률**. "이시 서른 두분"은 어휘적으로 어색한 조합이라 학습 말뭉치에서 거의 등장하지 않으므로 매우 낮은 확률.


### ❓ Q2-2. 순방향 / 역방향 / 마스크 언어 모델

> (a) GPT는 왜 **순방향**만 학습하는가? (생성 태스크의 특성과 연관지어)
>
> (b) BERT(MLM)가 **양방향**인 이유는 무엇인가? GPT 대비 어떤 장점이 있는가?
>
> (c) 세 방식 중 **문장 이해 능력**이 가장 뛰어난 것은? 그 이유는?
>
> (d) 세 방식 중 **다음 토큰 생성**에 가장 적합한 것은? 그 이유는?

**🔑 답:**
(a) GPT는 텍스트 생성 모델로, 사람이 글을 읽고 쓰는 자연스러운 방향(왼쪽→오른쪽)을 따른다. 생성 시점에는 미래 단어를 알 수 없으므로 과거 정보만으로 다음 단어를 예측해야 함.
(b) BERT는 [MASK] 토큰의 앞뒤 문맥을 모두 이용해 빈칸을 채우는 방식으로 학습. **앞뒤 문맥을 동시에 활용**할 수 있어 문장 이해(분류, NER, QA)에서 GPT보다 강력.
(c) **MLM(BERT)** — 양방향 문맥을 모두 활용하므로 단어의 의미를 더 풍부하게 표현. 분류·감성·NER 등 이해 태스크에서 우수.
(d) **순방향 LM(GPT)** — 다음 단어를 예측하는 방식 그대로 학습되어 자연스러운 텍스트 생성에 최적화. ChatGPT 같은 생성 모델의 기반.


### 📝 Q2-3. 빈칸 채우기 — KcBERT MLM 모델 로드

> **다음 코드 셀의 `______` 빈칸을 채우세요.**
> KcBERT의 MLM 모델을 불러와 추론 모드로 전환하고 디바이스로 옮기는 코드입니다.


In [2]:
# 🔑 Q2-3 (교수용 정답): KcBERT MLM 모델·토크나이저 로드
from transformers import BertTokenizer, BertForMaskedLM   # 🔑 (a) BertTokenizer (b) BertForMaskedLM
import torch
import torch.nn.functional as F

tokenizer  = BertTokenizer.from_pretrained("beomi/kcbert-base",   # 🔑 (c) BertTokenizer (d) beomi/kcbert-base
                                           do_lower_case=False)   # 🔑 (e) False (한국어는 대소문자 구분 X 의미 없음 → False 권장)
mlm_model  = BertForMaskedLM.from_pretrained("beomi/kcbert-base") # 🔑 (f) BertForMaskedLM (g) beomi/kcbert-base
mlm_model.eval()                                                  # 🔑 (h) eval — Dropout/BatchNorm을 추론 모드로
mlm_model.to(DEVICE)                                              # 🔑 (i) to — DEVICE로 이동

print("=" * 70)
print(f"  토크나이저 클래스: {type(tokenizer).__name__}")
print(f"  MLM 모델 클래스 : {type(mlm_model).__name__}")
print(f"  파라미터 수      : {sum(p.numel() for p in mlm_model.parameters()):,}")
print(f"  특수 토큰 ID     : [PAD]={tokenizer.pad_token_id}, "
      f"[CLS]={tokenizer.cls_token_id}, [SEP]={tokenizer.sep_token_id}, "
      f"[MASK]={tokenizer.mask_token_id}")
print("=" * 70)

# ─────────────────────────────────────────────────────────
#  (참고) 순방향 언어 모델 — MLM 헤드를 활용한 다음 토큰 후보 점수
# ─────────────────────────────────────────────────────────
sentence_natural = "오늘 점심으로 김치찌개를 [MASK]"
sentence_strange = "오늘 점심으로 김치찌개를 [MASK]했다"
for s in [sentence_natural, sentence_strange]:
    inputs = tokenizer(s, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = mlm_model(**inputs).logits
    mask_pos = (inputs.input_ids[0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
    if len(mask_pos) > 0:
        topk = torch.topk(F.softmax(logits[0, mask_pos[0]], dim=-1), 3)
        cands = [(tokenizer.convert_ids_to_tokens(i.item()), p.item())
                 for i, p in zip(topk.indices, topk.values)]
        print(f"\n  '{s}'")
        for tok, prob in cands:
            print(f"     → {tok:<10s} {prob*100:5.2f}%")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: beomi/kcbert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  토크나이저 클래스: BertTokenizer
  MLM 모델 클래스 : BertForMaskedLM
  파라미터 수      : 108,950,064
  특수 토큰 ID     : [PAD]=0, [CLS]=2, [SEP]=3, [MASK]=4

  '오늘 점심으로 김치찌개를 [MASK]'
     → 먹는다        24.83%
     → 건다          7.45%
     → 버렸다         3.80%

  '오늘 점심으로 김치찌개를 [MASK]했다'
     → 선물          6.41%
     → 구입          5.30%
     → 달라고         4.92%


### ❓ Q2-4. `eval()` 모드 vs `train()` 모드

> (a) `model.eval()`은 어떤 효과가 있나요? (적어도 두 가지 작동 변화)
>
> (b) 추론 시 `eval()` 호출을 잊으면 어떤 문제가 생기나요?
>
> (c) `with torch.no_grad():` 블록의 역할은 무엇인가요?

**🔑 답:**
(a) ① **Dropout 비활성화** — 학습 시 무작위 뉴런을 제거하던 레이어가 모두 통과시킴. ② **BatchNorm을 학습된 통계값으로 고정** — 미니배치 통계가 아닌 학습 시 누적된 running mean/var 사용.
(b) Dropout이 활성화된 채 추론하면 **결과가 매번 달라지는** 비결정적 출력이 나옴. 같은 입력에 대해 다른 임베딩이 추출되므로 평가 지표가 불안정해짐.
(c) 자동 미분(autograd) 그래프를 만들지 않아 **GPU 메모리 사용량이 크게 줄고 속도가 빨라짐**. 추론 시 gradient가 필요없으므로 항상 사용해야 함.


In [3]:
# ──────────────────────────────────────────────────────────────
# 함수 정의: 특정 단어가 그 위치에 올 확률 계산
#
# 사용 흐름:
#   1. 문장의 target_word를 [MASK]로 교체
#   2. 토크나이저로 input_ids 생성
#   3. [MASK] 위치 인덱스 찾기
#   4. BERT MLM에 통과시켜 logits 추출
#   5. softmax로 확률화
#   6. target_word에 해당하는 토큰 ID의 확률값 반환
# ──────────────────────────────────────────────────────────────

def compute_token_probability(sentence: str, target_word: str) -> float:
    """
    문장에서 target_word 위치를 [MASK]로 바꾼 뒤
    BERT MLM이 그 위치에 target_word를 예측할 확률을 반환합니다.
    MLM은 양방향 문맥을 모두 사용하는 BERT의 학습 방식입니다.
    """
    # 1) target_word를 첫 번째 등장 위치에서 [MASK]로 교체
    masked = sentence.replace(target_word, "[MASK]", 1)

    # 2) 토크나이저: 문자열 → input_ids 등 텐서 (PyTorch 형식)
    inputs = tokenizer(masked, return_tensors="pt").to(DEVICE)

    # 3) [MASK] 토큰의 위치 찾기
    #    tokenizer.mask_token_id는 [MASK]에 해당하는 토큰 ID (KcBERT에서는 4)
    mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
    if len(mask_idx) == 0:
        return 0.0

    # 4) MLM 추론: logits.shape = [batch=1, seq_len, vocab_size=30000]
    with torch.no_grad():
        logits = mlm_model(**inputs).logits

    # 5) [MASK] 위치만 추출 후 softmax → 확률 분포
    probs = F.softmax(logits[0, mask_idx[0]], dim=-1)

    # 6) target_word를 토큰화하여 첫 번째 토큰의 확률을 반환
    target_ids = tokenizer.encode(target_word, add_special_tokens=False)
    if not target_ids:
        return 0.0
    return probs[target_ids[0]].item()


# ── 자연스러운 문장 vs 부자연스러운 문장 비교 ────────────────
print("=" * 60)
print("  MLM으로 계산한 단어 등장 확률 비교")
print("  (같은 위치에 더 자연스러운 단어일수록 높은 확률)")
print("=" * 60)

test_cases = [
    # (문장, 비교할 단어1, 비교할 단어2)
    ("이 영화는 정말 재미있었어요",  "정말", "그냥"),
    ("난폭 운전은 매우 위험합니다",  "난폭", "무모"),
    ("오늘 날씨가 맑고 화창합니다",  "맑고", "좋고"),
]

for sentence, word1, word2 in test_cases:
    s1 = sentence
    s2 = sentence.replace(word1, word2, 1) if word1 in sentence else sentence

    p1 = compute_token_probability(s1, word1)
    p2 = compute_token_probability(s2, word2)

    winner = word1 if p1 >= p2 else word2
    print(f"\n  문장: {sentence}")
    print(f"    '{word1}' 확률: {p1:.6f}")
    print(f"    '{word2}' 확률: {p2:.6f}")
    print(f"    → 더 자연스러운 표현: '{winner}'")


  MLM으로 계산한 단어 등장 확률 비교
  (같은 위치에 더 자연스러운 단어일수록 높은 확률)

  문장: 이 영화는 정말 재미있었어요
    '정말' 확률: 0.429502
    '그냥' 확률: 0.001796
    → 더 자연스러운 표현: '정말'

  문장: 난폭 운전은 매우 위험합니다
    '난폭' 확률: 0.000034
    '무모' 확률: 0.000021
    → 더 자연스러운 표현: '난폭'

  문장: 오늘 날씨가 맑고 화창합니다
    '맑고' 확률: 0.000033
    '좋고' 확률: 0.000108
    → 더 자연스러운 표현: '좋고'


### 🧮 Q2-5. `compute_token_probability` 함수 분석

> (a) `tokenizer.mask_token_id`의 값은 KcBERT에서 무엇인가요? (vocab.txt 기준)
>
> (b) `logits.shape`이 `[1, seq_len, 30000]`이라면 30000은 무엇을 의미하는가?
>
> (c) `F.softmax(logits, dim=-1)`에서 `dim=-1`은 어느 축에 softmax를 적용하라는 뜻인가?
>
> (d) `with torch.no_grad():`가 없으면 어떤 부작용이 발생할까?

**🔑 답:**
(a) **4** — KcBERT의 vocab.txt에서 [PAD]=0, [UNK]=1, [CLS]=2, [SEP]=3, [MASK]=**4**
(b) **vocab_size** (어휘 집합 크기) — 각 위치마다 30,000개 단어 후보의 점수가 출력됨
(c) **마지막 축** (vocab_size 축, 즉 30000 차원) — 어휘 30,000개에 대한 확률 분포가 만들어지도록 합이 1이 되게 정규화
(d) ① **GPU 메모리 폭증** (autograd 그래프 누적)  ② **속도 저하**  ③ 추론에는 gradient가 필요 없으므로 항상 끄는 것이 표준


### 🐛 Q2-6. 오류 수정 — MLM 호출 (3개)

> ```python
> inputs = tokenizer(masked, return_tensors="np")          # 오류 1
> mask_idx = inputs['input_ids'].argmax()                   # 오류 2
> probs = F.softmax(mlm_model(**inputs), dim=-1)            # 오류 3
> ```

**🔑 답:**
1. `return_tensors="np"` → **`"pt"`** (PyTorch 텐서를 반환해야 BERT 모델에 입력 가능)
2. `argmax()`는 가장 큰 ID를 반환할 뿐 [MASK] 위치와 무관 → **`(inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]`** 로 [MASK] 토큰 위치를 직접 찾아야 함
3. `mlm_model(**inputs)`의 결과는 객체 → **`.logits` 속성**을 추출한 뒤 softmax 적용. 또한 특정 위치(`[0, mask_idx[0]]`)만 골라야 함


In [4]:
# ──────────────────────────────────────────────────────────────
# MLM 직접 체험 — [MASK] 위치 상위 10개 후보 예측
#
# BERT의 학습 방식 그대로:
#   앞뒤 문맥을 모두 보고 [MASK]에 올 가장 적절한 단어를 예측합니다.
#   이것이 GPT(단방향)와 BERT(양방향)의 핵심 차이입니다.
# ──────────────────────────────────────────────────────────────

def predict_masked_token(masked_sentence: str, top_k: int = 10):
    """
    [MASK] 위치에 올 단어를 상위 K개 확률과 함께 반환합니다.

    Args:
        masked_sentence: [MASK] 토큰을 포함한 문장
        top_k: 반환할 상위 후보 개수 (기본 10)

    Returns:
        [(token_str, prob), ...]  형식의 리스트
    """
    inputs   = tokenizer(masked_sentence, return_tensors="pt").to(DEVICE)
    mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

    with torch.no_grad():
        logits = mlm_model(**inputs).logits
    probs = F.softmax(logits[0, mask_idx[0]], dim=-1)

    # torch.topk: 상위 K개의 값과 인덱스를 동시에 반환
    top_vals, top_ids = torch.topk(probs, top_k)

    results = []
    for val, idx in zip(top_vals, top_ids):
        # convert_ids_to_tokens: 토큰 ID → 토큰 문자열
        token = tokenizer.convert_ids_to_tokens([idx.item()])[0]
        results.append((token, val.item()))
    return results


# 테스트 문장 — 직접 바꿔서 실험해보세요
test_sentences = [
    "어제 카페 갔었어 거기 [MASK] 많더라",
    "이 영화는 정말 [MASK]",
    "오늘 날씨가 [MASK] 화창합니다",
    "딥러닝을 배우는 것은 [MASK] 흥미롭습니다",
]

print("=" * 60)
print("  BERT MLM 예측 — [MASK] 위치 Top 10 후보")
print("  (앞뒤 문맥을 모두 보고 예측 ← 양방향의 힘)")
print("=" * 60)

for sent in test_sentences:
    print(f"\n문장: {sent}")
    preds = predict_masked_token(sent, top_k=10)
    print(f"  {'순위':>4}   {'예측 단어':15}   {'확률':>10}   {'막대그래프'}")
    print("  " + "─" * 55)
    for rank, (token, prob) in enumerate(preds, 1):
        bar = "█" * int(prob * 200)
        print(f"  {rank:4d}   {token:15}   {prob:10.6f}   {bar}")


  BERT MLM 예측 — [MASK] 위치 Top 10 후보
  (앞뒤 문맥을 모두 보고 예측 ← 양방향의 힘)

문장: 어제 카페 갔었어 거기 [MASK] 많더라
    순위   예측 단어                     확률   막대그래프
  ───────────────────────────────────────────────────────
     1   사람                  0.385594   █████████████████████████████████████████████████████████████████████████████
     2   사람들                 0.101325   ████████████████████
     3   엄청                  0.041183   ████████
     4   손님                  0.030261   ██████
     5   ##도                 0.029090   █████
     6   중국인                 0.028276   █████
     7   중국인들                0.016176   ███
     8   애들                  0.010576   ██
     9   아줌마들                0.010516   ██
    10   겁나                  0.009827   █

문장: 이 영화는 정말 [MASK]
    순위   예측 단어                     확률   막대그래프
  ───────────────────────────────────────────────────────
     1   싫다                  0.054187   ██████████
     2   대단하다                0.046948   █████████
     3   아니다                 0.041410

### ❓ Q2-7. `torch.topk` 함수

> (a) `torch.topk(probs, k=10)`이 반환하는 두 값은 무엇인가?
>
> (b) `convert_ids_to_tokens([4])`의 결과는?
>
> (c) Top-10 예측 결과의 확률을 모두 더하면 1이 되나요? 왜?

**🔑 답:**
(a) **`(values, indices)`** 튜플 — 상위 K개의 **확률값**과 그에 해당하는 **토큰 ID**
(b) **`["[MASK]"]`** — 토큰 ID 4번은 [MASK]
(c) **아니오, 1보다 작음.** 전체 어휘 30,000개에 대한 softmax 확률에서 상위 10개만 뽑은 것이므로 보통 합은 0.5~0.95 정도. 모두 더해 1이 되려면 30,000개를 모두 합해야 함.


### 🧮 Q2-8. 빈칸 채우기 — Top-K 추출

```python
def predict_masked_token(masked_sentence, top_k=10):
    inputs   = tokenizer(masked_sentence, return_tensors="____").to(DEVICE)  # (a)
    mask_idx = (inputs['input_ids'][0] == tokenizer.________).nonzero(as_tuple=True)[0]  # (b)
    with torch.________():                                                      # (c)
        logits = mlm_model(**inputs).________                                    # (d)
    probs    = F.________(logits[0, mask_idx[0]], dim=____)                     # (e)(f)
    top_vals, top_ids = torch.________(probs, top_k)                             # (g)
    return [(tokenizer.________(\[i.item()\])[0], v.item())
            for v, i in zip(top_vals, top_ids)]                                  # (h)
```

**🔑 답:**
(a) `pt`  (b) `mask_token_id`  (c) `no_grad`  (d) `logits`
(e) `softmax`  (f) `-1`  (g) `topk`  (h) `convert_ids_to_tokens`


---
## 3단계. 트랜스포머 구조 탐색 — `BertConfig`와 `BertModel`

### 🔬 KcBERT 모델 구조 한눈에 보기

```
입력 토큰 시퀀스 (input_ids)
    ↓
[임베딩 레이어]  ← 토큰 임베딩 + 위치 임베딩 + 세그먼트 임베딩
    ↓
[트랜스포머 블록 × 12]  ← Self-Attention + Feed-Forward Network
    ↓
출력 벡터 시퀀스 (각 토큰마다 768차원)
    ↓
[Pooler]  ← [CLS] 토큰 위치만 추출하여 fully-connected + tanh
    ↓
[CLS] 벡터 (768차원)  → 문장 분류, 유사도 계산에 활용
```

### 📌 핵심 클래스

| 클래스 | 역할 |
|--------|------|
| `BertConfig` | 모델 **설계 설정값**만 담은 객체 (레이어 수, 헤드 수, hidden 크기, vocab 크기 등). 가중치는 없음. |
| `BertModel` | 실제 **파라미터를 가진 모델 객체**. forward 가능. |
| `BertForMaskedLM` | `BertModel` + MLM 예측 헤드. (지난 절에서 사용) |
| `BertTokenizer` | 텍스트 → input_ids 변환기 |

### 🧮 BERT-base 핵심 하이퍼파라미터

| 설정 | 값 | 의미 |
|------|-----|------|
| `vocab_size` | 30,000 | 어휘 집합 크기 |
| `hidden_size` | **768** | 각 토큰을 표현하는 벡터의 차원 |
| `num_hidden_layers` | **12** | 트랜스포머 블록(레이어) 수 |
| `num_attention_heads` | **12** | Self-Attention 헤드 수 |
| `intermediate_size` | 3072 | Feed-Forward 중간층 차원 (보통 4×hidden) |
| `max_position_embeddings` | 300 | 처리 가능한 최대 토큰 수 |


### ❓ Q3-1. `BertConfig` vs `BertModel`

> (a) `BertConfig.from_pretrained("beomi/kcbert-base")`는 무엇을 반환하는가? 메모리는 얼마나 사용?
>
> (b) `BertModel.from_pretrained("beomi/kcbert-base")`는 무엇을 반환하는가? 어디에서 가중치를 받아오는가?
>
> (c) `BertModel`만 있어도 모델 학습/추론이 가능한 이유는? `BertConfig`만으로는 왜 부족한가?

**🔑 답:**
(a) **모델 설계 설정값**만 담은 가벼운 객체 (수 KB). hidden_size, num_layers 등 하이퍼파라미터만 들어있음.
(b) **실제 가중치(약 440MB)와 구조를 모두 가진 모델 객체**. HuggingFace Hub에서 `pytorch_model.bin` (또는 `.safetensors`) 파일을 다운받아 PyTorch 텐서로 로드.
(c) `BertModel`은 모든 weight 행렬과 bias 벡터가 **사전학습된 값으로 채워진 상태**라 바로 추론 가능. `BertConfig`만으로는 구조만 알 뿐 학습된 weight가 없어 출력이 무의미.


### 🧮 Q3-2. BERT 파라미터 수 직접 계산

> BERT-base의 임베딩 레이어와 트랜스포머 블록 한 개의 파라미터 수를 손으로 계산하세요.
>
> (a) 토큰 임베딩 행렬: `[vocab_size, hidden] = [30000, 768]` → 파라미터 수 = ______
>
> (b) 위치 임베딩 행렬: `[max_position, hidden] = [300, 768]` → 파라미터 수 = ______
>
> (c) Self-Attention의 Q/K/V 가중치 (각 `[768, 768]`): 파라미터 수 = ______ × 3 = ______
>
> (d) Self-Attention 출력 가중치 `[768, 768]`: ______
>
> (e) FFN 첫 번째 레이어 `[768, 3072]`: ______
>
> (f) FFN 두 번째 레이어 `[3072, 768]`: ______
>
> (g) 트랜스포머 블록 1개의 가중치 합계 (대략, bias·norm 제외): ______
>
> (h) 12개 블록 합계: ______ × 12 = ______

**🔑 답:**
(a) 30000 × 768 = **23,040,000** (~23M)
(b) 300 × 768 = **230,400** (~0.23M)
(c) 768 × 768 = **589,824**, × 3 = **1,769,472** (~1.77M)
(d) 768 × 768 = **589,824** (~0.59M)
(e) 768 × 3072 = **2,359,296** (~2.36M)
(f) 3072 × 768 = **2,359,296** (~2.36M)
(g) 약 1.77M + 0.59M + 2.36M + 2.36M = **약 7.08M** (블록 1개)
(h) 7.08M × 12 = **약 85M** (트랜스포머 블록 합계)

전체 = 임베딩 약 23M + 블록 약 85M + 기타 ≈ **약 110M (1.1억 개)** ✓


### 📝 Q3-3. 빈칸 채우기 — KcBERT 모델 구조 출력

> **다음 코드 셀의 `______` 빈칸을 채우세요.**
> `BertConfig`와 `BertModel`을 직접 불러와 KcBERT의 하이퍼파라미터를 확인합니다.


In [8]:
# 🔑 Q3-3 (교수용 정답): BertConfig / BertModel을 통한 구조 확인
from transformers import BertConfig, BertModel    # 🔑 (a) BertConfig (b) BertModel

config = BertConfig.from_pretrained("beomi/kcbert-base")              # 🔑 (c) BertConfig
model_bert = BertModel.from_pretrained("beomi/kcbert-base", config=config, attn_implementation="eager")  # 🔑 (d) BertModel (e) config
model_bert.eval()                                                       # 🔑 (f) eval
model_bert.to(DEVICE)                                                   # 🔑 (g) to

print("=" * 70)
print("  KcBERT 하이퍼파라미터")
print("=" * 70)
print(f"  hidden_size            : {config.hidden_size}")            # 768
print(f"  num_hidden_layers      : {config.num_hidden_layers}")      # 12
print(f"  num_attention_heads    : {config.num_attention_heads}")    # 12
print(f"  intermediate_size (FFN): {config.intermediate_size}")      # 3072
print(f"  vocab_size             : {config.vocab_size}")             # ~30,000
print(f"  max_position_embeddings: {config.max_position_embeddings}")# 300
print(f"  파라미터 총합          : {sum(p.numel() for p in model_bert.parameters()):,}")  # ~110M
print("=" * 70)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  KcBERT 하이퍼파라미터
  hidden_size            : 768
  num_hidden_layers      : 12
  num_attention_heads    : 12
  intermediate_size (FFN): 3072
  vocab_size             : 30000
  max_position_embeddings: 300
  파라미터 총합          : 108,918,528


### 🔢 Q3-4. 출력 예측

> 위 셀을 실행하면 다음과 같은 결과가 나옵니다. 빈칸을 채우세요.
>
> ```
> hidden_size                     ____    ← 각 토큰 표현 벡터의 차원 수
> num_hidden_layers               ____    ← 트랜스포머 블록(레이어) 수
> num_attention_heads             ____    ← Self-Attention 헤드 수
> intermediate_size               ____    ← Feed-Forward 중간 레이어 차원
>
> 전체 파라미터    :  약 _________ 개  (~110M)
> ```

**🔑 답:**
- hidden_size = **768**
- num_hidden_layers = **12**
- num_attention_heads = **12**
- intermediate_size = **3072** (= 4 × 768)
- 전체 파라미터 약 **108,000,000~110,000,000** 개 (~108M~110M)


### 🐛 Q3-5. 오류 수정 — BertModel 로드 (3개)

> ```python
> from transformers import BertConfig, GPT2Model           # 오류 1
> config = BertConfig.from_pretrained("kcbert-base")        # 오류 2
> model = BertModel(config=config)                          # 오류 3 (가중치 어떻게?)
> ```

**🔑 답:**
1. `GPT2Model` → **`BertModel`** (BERT 가중치를 GPT 모델에 로드 불가)
2. `"kcbert-base"` → **`"beomi/kcbert-base"`** (HuggingFace Hub 모델 ID 형식: `username/model-name`)
3. `BertModel(config=config)`은 **랜덤 초기화** 된 모델을 만듦 → **`BertModel.from_pretrained("beomi/kcbert-base")`** 로 사전학습 가중치 로드


### ❓ Q3-6. `attn_implementation="eager"`의 역할

> (a) `attn_implementation="eager"`와 `"sdpa"`의 차이는 무엇인가?
>
> (b) Self-Attention 가중치(가중합 계수)를 추출하려면 어느 쪽을 써야 하는가?
>
> (c) 추론 속도가 더 빠른 쪽은? 그 이유는?

**🔑 답:**
(a) `eager`: 어텐션을 단계별로 풀어서 직접 계산하는 표준 구현. attention weight 행렬을 저장 가능.
    `sdpa` (Scaled Dot Product Attention): PyTorch 2.0+의 최적화된 fused 커널. 빠르지만 attention weight를 반환하지 않음.
(b) **`eager`** — `output_attentions=True` 옵션이 작동하려면 eager 구현이 필요
(c) **`sdpa`** — Flash Attention 등 최적화 커널을 활용하여 메모리도 적게 쓰고 속도도 빠름. 단, attention weight 시각화는 불가


In [9]:
# ──────────────────────────────────────────────────────────────
# 트랜스포머 블록 내부 구조 상세 확인
#
# 각 트랜스포머 블록(레이어)은 다음으로 구성됩니다:
#   1. Self-Attention: 토큰 간 관계 포착 (12개 헤드 병렬)
#      - query, key, value (각각 [768, 768] 가중치)
#      - 어텐션 출력 (또 다른 [768, 768] 가중치)
#   2. Add & Norm: 잔차 연결(residual) + 레이어 정규화(LayerNorm)
#   3. Feed-Forward: 비선형 변환 (768 → 3072 → 768)
#   4. Add & Norm: 잔차 연결 + 레이어 정규화
#
# 핵심 메서드:
#   .named_modules(): 모델 내 모든 모듈을 (name, module)로 반환 (재귀적)
#   .encoder.layer[i]: i번째 트랜스포머 블록 객체
# ──────────────────────────────────────────────────────────────
print("=" * 70)
print("  트랜스포머 블록 구조 (1번 레이어 기준)")
print("=" * 70)

layer_0 = model_bert.encoder.layer[0]   # 첫 번째 트랜스포머 블록
for name, module in layer_0.named_modules():
    # weight 속성을 가진 leaf 모듈만 출력 (Linear, LayerNorm 등)
    if hasattr(module, 'weight') and len(list(module.children())) == 0:
        n = sum(p.numel() for p in module.parameters())
        print(f"  {name:50}  {n:>10,} 개")

print()
print("  [Head당 차원 계산]")
head_dim = config.hidden_size // config.num_attention_heads
print(f"  hidden_size ({config.hidden_size}) ÷ num_heads ({config.num_attention_heads}) = {head_dim}차원/헤드")
print(f"  → 각 헤드는 독립적으로 {head_dim}차원에서 어텐션을 계산합니다.")
print(f"  → 12개 헤드 결과를 이어 붙이면 다시 {config.hidden_size}차원이 됩니다.")


  트랜스포머 블록 구조 (1번 레이어 기준)
  attention.self.query                                   590,592 개
  attention.self.key                                     590,592 개
  attention.self.value                                   590,592 개
  attention.output.dense                                 590,592 개
  attention.output.LayerNorm                               1,536 개
  intermediate.dense                                   2,362,368 개
  output.dense                                         2,360,064 개
  output.LayerNorm                                         1,536 개

  [Head당 차원 계산]
  hidden_size (768) ÷ num_heads (12) = 64차원/헤드
  → 각 헤드는 독립적으로 64차원에서 어텐션을 계산합니다.
  → 12개 헤드 결과를 이어 붙이면 다시 768차원이 됩니다.


### 🧮 Q3-7. 헤드 차원 계산

> (a) `hidden_size = 768`, `num_attention_heads = 12`일 때, 헤드 1개의 차원은?
>
> (b) 헤드 1개당 Q/K/V 가중치 행렬의 shape은? (한 번 forward 시)
>
> (c) 12개 헤드의 출력을 이어 붙이면(concat) 차원이 다시 ____가 된다.
>
> (d) 만약 `num_attention_heads = 16`이라면 헤드당 차원은 얼마인가? (768이 16으로 나누어 떨어지는가?)

**🔑 답:**
(a) **768 ÷ 12 = 64차원/헤드**
(b) 입력 토큰 표현 [seq, 768] → 헤드별 Q/K/V는 각 [seq, **64**]. 단, 구현상 12개 헤드를 한꺼번에 [seq, 768] 형태로 계산한 뒤 head별로 reshape: `[seq, 12, 64]`
(c) 12 × 64 = **768** (concat 후 원래 차원으로 복원)
(d) 768 ÷ 16 = **48차원/헤드**. 768은 16으로 나누어 떨어짐(48). 일반적으로 hidden_size는 num_heads로 나누어 떨어져야 함.


### ❓ Q3-8. 잔차 연결(Residual Connection)과 레이어 정규화(LayerNorm)

> (a) 잔차 연결의 수식: `output = ______ + ______` 의 형태로 작성
>
> (b) 잔차 연결을 사용하는 이유 한 가지는? (gradient 관점에서)
>
> (c) LayerNorm은 어느 축(axis)을 따라 정규화를 수행하는가? (BatchNorm과의 차이)

**🔑 답:**
(a) `output = SubLayer(x) + x` — Self-Attention(x) + x, 또는 FFN(x) + x
(b) 깊은 네트워크에서 **gradient vanishing 문제 완화**. 역전파 시 gradient가 잔차 경로(`+x`)를 따라 손실 없이 앞쪽 레이어로 전달됨.
(c) **각 토큰(샘플) 내부의 feature 축**을 따라 정규화. BatchNorm이 batch 차원을 따라 평균/표준편차를 구하는 것과 달리, LayerNorm은 batch와 무관하게 동작 → **batch size에 영향을 받지 않음** (NLP에서 가변 길이 시퀀스 처리에 유리).


---
## 4단계. BERT 임베딩 추출 — `last_hidden_state`와 `pooler_output`

### 📐 BertModel의 핵심 출력

| 속성 | shape | 의미 | 활용 |
|------|-------|------|------|
| `last_hidden_state` | `[batch, seq_len, hidden]` | 마지막 레이어의 모든 토큰 벡터 | NER, 품사 태깅, 토큰 분류 |
| `pooler_output` | `[batch, hidden]` | [CLS] 토큰 → FC + tanh | 문장 분류, 유사도 |
| `hidden_states` | tuple of 13 tensors | 모든 레이어의 출력 (임베딩 + 12 블록) | 레이어별 분석 |
| `attentions` | tuple of 12 tensors | 각 레이어의 어텐션 가중치 `[batch, heads, seq, seq]` | 어텐션 시각화 |

### 🔬 처리 흐름

```
"안녕하세요 반갑습니다"
    ↓ tokenizer(...)
{input_ids:        [batch, seq_len],   # 토큰 ID
 attention_mask:   [batch, seq_len],   # 1=실제 토큰, 0=패딩
 token_type_ids:   [batch, seq_len]}   # 모두 0 (단일 문장)
    ↓ model_bert(**inputs)
{last_hidden_state: [batch, seq_len, 768],
 pooler_output:     [batch, 768]}
```


### ❓ Q4-1. BERT 입력 형식

> (a) BERT 토크나이저가 자동으로 추가하는 두 가지 특수 토큰은?
>
> (b) `attention_mask`에서 0과 1은 각각 무엇을 의미하는가?
>
> (c) `token_type_ids`가 모두 0일 때, 입력은 단일 문장인가 두 문장인가?

**🔑 답:**
(a) **`[CLS]` (앞)**, **`[SEP]` (뒤)** — `[CLS] 문장 [SEP]` 형식
(b) **0 = 패딩 토큰** ([PAD], 모델이 무시), **1 = 실제 토큰** (어텐션 계산에 참여)
(c) **단일 문장** — 두 문장 입력 시 두 번째 문장 영역의 token_type_ids는 1


### 📝 Q4-2. 빈칸 채우기 — BERT 임베딩 추출

> **다음 코드 셀의 `______` 빈칸을 채우세요.**
> 토크나이즈 → 모델 추론 → `last_hidden_state` / `pooler_output` 추출까지의 전체 흐름입니다.


In [10]:
# 🔑 Q4-2 (교수용 정답): BERT 임베딩 추출
sentence = "딥러닝 자연어 처리 수업이 흥미롭습니다"

# 1) 토크나이즈
inputs = tokenizer(sentence, return_tensors="pt")          # 🔑 (a) pt
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

# 2) 모델 forward (gradient 비활성화)
with torch.no_grad():                                      # 🔑 (b) no_grad
    outputs = model_bert(**inputs,
                         output_hidden_states=True,        # 🔑 (c) True
                         output_attentions=True)           # 🔑 (d) True

# 3) 출력 분리
last_hidden = outputs.last_hidden_state                    # 🔑 (e) last_hidden_state — [B, L, 768]
pooler      = outputs.pooler_output                        # 🔑 (f) pooler_output     — [B, 768] = Tanh(W·CLS)
all_hidden  = outputs.hidden_states                        # 🔑 (g) hidden_states     — len 13 (embed + 12 layers)
all_attn    = outputs.attentions                           # 🔑 (h) attentions        — len 12

print(f"입력 문장: {sentence}")
print(f"토큰 수    : {inputs['input_ids'].shape[1]}")
print(f"last_hidden_state shape: {tuple(last_hidden.shape)}")
print(f"pooler_output     shape: {tuple(pooler.shape)}")
print(f"hidden_states     수    : {len(all_hidden)}  (embedding + 12 layers)")
print(f"attentions        수    : {len(all_attn)}  (12 layers)")
print(f"한 레이어의 attention shape: {tuple(all_attn[0].shape)}  # [batch, heads, seq, seq]")


입력 문장: 딥러닝 자연어 처리 수업이 흥미롭습니다
토큰 수    : 14
last_hidden_state shape: (1, 14, 768)
pooler_output     shape: (1, 768)
hidden_states     수    : 13  (embedding + 12 layers)
attentions        수    : 12  (12 layers)
한 레이어의 attention shape: (1, 12, 14, 14)  # [batch, heads, seq, seq]


### 🔢 Q4-3. Shape 직접 계산

> 위 셀의 출력 shape을 예측하세요. (`batch=2`, `max_length=20`, `hidden=768`, `heads=12`, `layers=12`)
>
> (a) `outputs.last_hidden_state.shape` = `[__, __, __]`
>
> (b) `outputs.pooler_output.shape` = `[__, __]`
>
> (c) `len(outputs.hidden_states)` = `__`  (왜 12가 아니라 13인가?)
>
> (d) `outputs.hidden_states[0].shape` = `[__, __, __]`
>
> (e) `len(outputs.attentions)` = `__`
>
> (f) `outputs.attentions[0].shape` = `[__, __, __, __]`

**🔑 답:**
(a) `[2, 20, 768]` — [batch, seq_len, hidden]
(b) `[2, 768]` — [batch, hidden]
(c) **13** — `hidden_states[0]`은 트랜스포머 블록 통과 전 **임베딩 레이어 출력**, `hidden_states[1]`~`[12]`는 12개 트랜스포머 블록의 출력
(d) `[2, 20, 768]` — 모든 hidden_states는 같은 shape
(e) **12** — 트랜스포머 블록 수와 동일
(f) `[2, 12, 20, 20]` — `[batch, num_heads, seq_len, seq_len]` (Query × Key 어텐션 행렬)


### ❓ Q4-4. `last_hidden_state` vs `pooler_output`

> (a) `last_hidden_state[:, 0, :]`이 의미하는 것은? `pooler_output`과 어떻게 다른가?
>
> (b) `pooler_output`은 `last_hidden_state[:, 0, :]`에 추가로 어떤 연산이 적용되는가?
>
> (c) 문장 분류 태스크에서는 보통 둘 중 어느 쪽을 사용하는가?
>
> (d) 토큰 분류(NER, POS 태깅) 태스크에서는 어느 쪽을 사용하는가?

**🔑 답:**
(a) `last_hidden_state[:, 0, :]`은 **마지막 트랜스포머 블록 통과 후의 [CLS] 위치 벡터** (raw). `pooler_output`은 여기에 추가 변환을 적용한 것.
(b) **Linear(768→768) + tanh** — `[CLS]` 위치 벡터를 fully-connected layer + tanh 활성화로 변환. NSP(Next Sentence Prediction) 학습 시 사용된 헤드.
(c) 일반적으로 **둘 다 사용 가능**하나, 실무에서는 `last_hidden_state[:, 0, :]`을 직접 사용하는 경우가 많음 (pooler는 NSP에 특화되어 분류에 최적이 아닐 수 있음).
(d) **`last_hidden_state`** — 각 토큰별 분류가 필요하므로 모든 위치의 벡터를 사용.


### 🐛 Q4-5. 오류 수정 — 임베딩 추출 (3개)

> ```python
> features = tokenizer(sentences, padding=True)              # 오류 1
> with torch.enable_grad():                                  # 오류 2
>     outputs = model_bert(features)                          # 오류 3
> print(outputs.cls_output.shape)
> ```

**🔑 답:**
1. `padding=True`만 있고 `return_tensors`가 없어 dict 안 값이 list — **`return_tensors="pt"`** 추가
2. `enable_grad()`는 학습 시 사용. 추론에서는 **`torch.no_grad()`** 사용
3. `model_bert(features)`는 dict 객체를 통째로 넘김 → **`model_bert(**features)`** 로 키워드 인자로 펼쳐 전달
4. (덤) `outputs.cls_output`은 존재하지 않는 속성 → **`outputs.pooler_output`** 또는 **`outputs.last_hidden_state[:, 0, :]`**


In [11]:
# ──────────────────────────────────────────────────────────────
# 코사인 유사도로 문장 의미 유사성 측정
#
# [CLS] 벡터는 문장 전체의 의미를 768차원 벡터 하나로 압축합니다.
# 두 [CLS] 벡터 사이의 코사인 유사도는 두 문장의 의미적 유사성을 나타냅니다.
#
# 코사인 유사도 = (v1 · v2) / (|v1| × |v2|)
#   값 범위: -1 (반대) ~ +1 (동일 방향)
#   문장 임베딩에서는 보통 0.5~0.99 범위에 분포
# ──────────────────────────────────────────────────────────────
import torch.nn.functional as F

def get_sentence_embedding(sentence: str) -> torch.Tensor:
    """문장 → [CLS] 벡터(768차원) 반환"""
    inputs = tokenizer(
        sentence, return_tensors="pt",
        max_length=128, truncation=True, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        out = model_bert(**inputs)
    return out.pooler_output[0]   # [768]

def cosine_similarity(v1: torch.Tensor, v2: torch.Tensor) -> float:
    """두 벡터의 코사인 유사도 (-1 ~ +1)"""
    # F.cosine_similarity는 batch 차원이 필요하므로 unsqueeze(0)으로 추가
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()


# 문장 유사도 실험
sentence_groups = [
    ("이 영화 정말 재미있었어요",     "이 영화는 너무 재밌어서 또 보고 싶어요",  "높음"),
    ("이 영화 정말 재미있었어요",     "오늘 날씨가 맑고 화창하네요",             "낮음"),
    ("오늘 점심 뭐 먹을까",          "저녁 메뉴 고르기가 어렵네",               "중간"),
    ("딥러닝 모델을 학습시킵니다",    "인공신경망을 훈련합니다",                 "높음"),
]

print("=" * 75)
print("  문장 수준 임베딩 코사인 유사도 ([CLS] 벡터 기반)")
print("=" * 75)

for s1, s2, expected in sentence_groups:
    v1  = get_sentence_embedding(s1)
    v2  = get_sentence_embedding(s2)
    sim = cosine_similarity(v1, v2)
    bar_len = int(sim * 40)
    bar = "█" * bar_len + "░" * (40 - bar_len)

    print(f"\n  문장1: {s1}")
    print(f"  문장2: {s2}")
    print(f"  유사도: {sim:.4f}  [{bar}]  (예상: {expected})")


  문장 수준 임베딩 코사인 유사도 ([CLS] 벡터 기반)

  문장1: 이 영화 정말 재미있었어요
  문장2: 이 영화는 너무 재밌어서 또 보고 싶어요
  유사도: 0.9507  [██████████████████████████████████████░░]  (예상: 높음)

  문장1: 이 영화 정말 재미있었어요
  문장2: 오늘 날씨가 맑고 화창하네요
  유사도: 0.8134  [████████████████████████████████░░░░░░░░]  (예상: 낮음)

  문장1: 오늘 점심 뭐 먹을까
  문장2: 저녁 메뉴 고르기가 어렵네
  유사도: 0.7558  [██████████████████████████████░░░░░░░░░░]  (예상: 중간)

  문장1: 딥러닝 모델을 학습시킵니다
  문장2: 인공신경망을 훈련합니다
  유사도: 0.8432  [█████████████████████████████████░░░░░░░]  (예상: 높음)


### 🧮 Q4-6. 코사인 유사도 직접 계산

> 두 벡터 `v1 = [3, 4]`, `v2 = [4, 3]`이 주어졌을 때:
>
> (a) 내적(dot product) `v1 · v2` = ______
>
> (b) `|v1|` = ______, `|v2|` = ______
>
> (c) 코사인 유사도 = `____ / (____ × ____)` = ______
>
> (d) 두 벡터가 정확히 같은 방향이면 코사인 유사도는?
>
> (e) 두 벡터가 직교하면(서로 90°)?

**🔑 답:**
(a) v1·v2 = 3×4 + 4×3 = **24**
(b) |v1| = √(9+16) = √25 = **5**, |v2| = √(16+9) = √25 = **5**
(c) 24 / (5 × 5) = **24/25 = 0.96**
(d) **+1** (cos 0° = 1)
(e) **0** (cos 90° = 0)


### ❓ Q4-7. `F.cosine_similarity`와 `unsqueeze(0)`

> (a) `F.cosine_similarity(v1, v2)`에 `[768]` shape 텐서를 직접 넣으면 왜 에러가 나는가?
>
> (b) `v1.unsqueeze(0)`은 무엇을 하는가? shape이 어떻게 변하는가?
>
> (c) 768차원 벡터를 batch=1로 만들면 shape은 `[__, __]`이 된다.

**🔑 답:**
(a) `F.cosine_similarity`는 기본적으로 `dim=1` 축을 따라 계산하므로 **2D 이상 텐서**를 기대. 1D 텐서를 넣으면 dim=1이 존재하지 않아 에러.
(b) `unsqueeze(dim)`: 지정한 dim에 크기 1짜리 새 축을 삽입. `[768]` → `[1, 768]`
(c) `[1, 768]` (batch=1, hidden=768)


### 📝 Q4-8. 빈칸 채우기 — `get_sentence_embedding` 함수 (복습)

> 위에서 사용했던 `get_sentence_embedding` 함수를 **복습 차원에서 직접 다시 작성**해 봅시다.
> 다음 코드 셀의 `______` 빈칸을 채워 함수를 재정의하고, 한 문장으로 동작을 확인하세요.


In [12]:
# 🔑 Q4-8 (교수용 정답): get_sentence_embedding 함수 재정의
def get_sentence_embedding(sentence: str) -> torch.Tensor:
    """문장 → [CLS]에서 유래한 pooler_output(768차원) 반환"""
    inputs = tokenizer(
        sentence,
        return_tensors="pt",                # 🔑 (a) pt
        max_length=128,                     # 🔑 (b) 128
        truncation=True,                    # 🔑 (c) True
        padding=True,                       # 🔑 (d) True
    ).to(DEVICE)
    with torch.no_grad():                   # 🔑 (e) no_grad
        out = model_bert(**inputs)
    return out.pooler_output[0]             # 🔑 (f) pooler_output

# 동작 확인
v = get_sentence_embedding("이 영화 정말 재미있었어요")
print(f"✅ 임베딩 shape: {tuple(v.shape)}   (예상: (768,))")


✅ 임베딩 shape: (768,)   (예상: (768,))


---
## 5단계. Self-Attention 시각화

### 🔬 Self-Attention 핵심 개념 복습

Self-Attention은 시퀀스의 **각 토큰이 다른 모든 토큰에 얼마나 집중하는지**를 계산합니다.

```
"이 영화는 정말 재밌어요"
→ '재밌어요'를 이해할 때 '영화', '정말'에 높은 어텐션 가중치 부여
→ '는', '이' 같은 조사에는 낮은 가중치
```

### 🧮 Q/K/V 수식 (강의노트 핵심)

각 토큰 벡터 $x$에서 세 가지를 만들어냅니다:

$$Q = x W^Q, \quad K = x W^K, \quad V = x W^V$$

어텐션 가중치는 다음과 같이 계산됩니다:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| 구성요소 | 의미 |
|---------|------|
| $Q$ (Query) | "내가 무엇에 집중하고 싶은가" — 검색 질의 |
| $K$ (Key) | "나는 어떤 정보를 가지고 있는가" — 색인 |
| $V$ (Value) | "그 정보의 실제 내용" — 결과값 |
| $d_k$ | 헤드당 차원 (KcBERT는 64) — gradient 안정화 |

### 📊 어텐션 가중치 행렬 shape

```
attentions[layer]: [batch, num_heads, seq_len, seq_len]
                                       ↑          ↑
                                     Query 토큰   Key 토큰
                                  (질의 위치)   (참조 위치)

attentions[layer][0, h, i, j] = "h번째 헤드에서, 토큰 i가 토큰 j에 집중하는 정도"
```

12개 헤드는 각자 다른 언어적 패턴을 학습합니다:
- 일부 헤드: 구문 구조 (주어-서술어 관계)
- 일부 헤드: 지시어 해소 (이것 → 영화)
- 일부 헤드: 인접 토큰 (바로 옆 단어 집중)


### ❓ Q5-1. Q/K/V 개념 정리

> (a) Query, Key, Value를 도서관 검색에 비유해서 설명하세요.
>
> (b) `softmax(QK^T)` 계산에서 `^T`는 무엇을 의미하는가? (행렬 연산)
>
> (c) `QK^T` 행렬의 shape은 `[seq_len, seq_len]`. 행과 열은 각각 무엇을 의미하는가?
>
> (d) 왜 `QK^T`를 `√d_k`로 나누는가? (KcBERT는 √64 = 8)

**🔑 답:**
(a) **Query** = 검색 질의문 ("머신러닝 책 어디 있어?"), **Key** = 책의 색인/제목 (검색 매칭 대상), **Value** = 책의 실제 내용 (검색 결과로 가져올 정보)
(b) **전치 행렬(transpose)** — 행과 열을 바꿈. K가 `[seq, d]`면 K^T는 `[d, seq]`
(c) 행 = **Query 토큰** (질의 위치), 열 = **Key 토큰** (참조 위치). [i, j] 값은 토큰 i가 토큰 j에 얼마나 집중하는지
(d) 차원이 커질수록 내적값이 커져 softmax가 매우 sharp해지고 **gradient가 소실**됨. √d_k로 나눠 분산을 1로 정규화하여 학습 안정성 확보 → **Scaled Dot-Product Attention**


### 🧮 Q5-2. 가중합(Weighted Sum) 계산 (강의노트 수식 3-6)

> 강의노트의 예시: '카페' 토큰에 대한 어텐션 가중치가 다음과 같다고 하자.
>
> | 키 토큰 | 가중치 |
> |---------|--------|
> | 어제   | 0.1   |
> | 카페   | 0.1   |
> | 갔었어 | 0.2   |
> | 거기   | 0.4   |
> | 사람   | 0.1   |
> | 많더라 | 0.1   |
>
> (a) 가중치들의 합 = ______ (반드시 1이 되어야 함)
>
> (b) 만약 V_거기 = `[1.0]`, V_카페 = `[2.0]`이고 다른 V는 모두 `[0.0]`이라면, Z_카페 = ______
>
> (c) 결국 셀프 어텐션의 결과 Z_카페는 어떤 토큰의 V를 가장 많이 반영하는가?

**🔑 답:**
(a) 0.1 + 0.1 + 0.2 + 0.4 + 0.1 + 0.1 = **1.0** ✓ (softmax이므로 항상 1)
(b) Z_카페 = 0.1×0 + 0.1×2.0 + 0.2×0 + 0.4×1.0 + 0.1×0 + 0.1×0 = **0.6**
(c) **'거기' (가중치 0.4)** — 가중치가 가장 큰 토큰의 V가 최종 Z에 가장 큰 영향


### 📝 Q5-3. 빈칸 채우기 — 어텐션 가중치 추출

> **다음 코드 셀의 `______` 빈칸을 채우세요.**
> `output_attentions=True`로 모델을 호출해 12개 레이어의 어텐션 가중치를 꺼내는 코드입니다.


In [13]:
# 🔑 Q5-3 (교수용 정답): 어텐션 가중치 추출
sentence_attn = "딥러닝 자연어 처리 수업이 흥미롭습니다"

inputs_attn = tokenizer(sentence_attn, return_tensors="pt").to(DEVICE)   # 🔑 (a) pt
tokens      = tokenizer.convert_ids_to_tokens(inputs_attn["input_ids"][0])

with torch.no_grad():                                  # 🔑 (b) no_grad
    outputs_attn = model_bert(
        **inputs_attn,
        output_attentions=True,                        # 🔑 (c) True
    )

# attentions: tuple of length 12, 각 원소 shape = [batch, heads, seq, seq]
attentions = outputs_attn.attentions                   # 🔑 (d) 12 (e) attentions

print(f"입력 토큰     : {tokens}")
print(f"레이어 수     : {len(attentions)}")
print(f"한 레이어 shape: {tuple(attentions[0].shape)}")

# 마지막 레이어, 모든 헤드 평균 → [seq, seq]
attn_last = attentions[-1][0].mean(dim=0)              # 🔑 (f) -1 (g) 0 (헤드 차원이 0번 axis)
print(f"\n어텐션 행렬 shape (헤드 평균 후): {tuple(attn_last.shape)}")
print(f"각 행의 합 (반드시 1.0이어야 함): {attn_last.sum(dim=-1)}")


입력 토큰     : ['[CLS]', '딥', '##러', '##닝', '자연', '##어', '처리', '수업', '##이', '흥', '##미', '##롭', '##습니다', '[SEP]']
레이어 수     : 12
한 레이어 shape: (1, 12, 14, 14)

어텐션 행렬 shape (헤드 평균 후): (14, 14)
각 행의 합 (반드시 1.0이어야 함): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


### 🔢 Q5-4. 어텐션 행렬 인덱싱

> `out.attentions[11][0, 7]`의 의미를 분해해 설명하세요.
>
> (a) `out.attentions[11]`: ____ 번째 트랜스포머 레이어의 어텐션 → shape `[__, __, __, __]`
>
> (b) `out.attentions[11][0]`: 0번 ____ 의 어텐션 → shape `[__, __, __]`
>
> (c) `out.attentions[11][0, 7]`: 0번 배치, 7번 ____ 의 어텐션 → shape `[__, __]`
>
> (d) `out.attentions[11][0, 7][2, 5]`: 토큰 ____ 가 토큰 ____ 에 집중하는 정도

**🔑 답:**
(a) **12번째 (마지막)** 트랜스포머 레이어 → `[batch, num_heads, seq_len, seq_len]`
(b) **배치(batch)** → `[num_heads, seq_len, seq_len]`
(c) **헤드(head)** → `[seq_len, seq_len]`
(d) 토큰 **2번**(Query)이 토큰 **5번**(Key)에 집중하는 정도 (스칼라 값, 0~1 사이)


### ❓ Q5-5. 어텐션 행렬의 행 합

> (a) `attn[i].sum()` (한 행의 합)은 항상 어떤 값에 가까운가? 왜?
>
> (b) `attn[:, j].sum()` (한 열의 합)은 어떤가? 1이 되는가?
>
> (c) 만약 `attn[3, 5] = 0.8`이고 다른 `attn[3, j]`들이 모두 작다면, 토큰 3에 대해 어떤 해석이 가능한가?

**🔑 답:**
(a) **항상 1.0**. softmax가 행 단위로 적용되므로 각 Query 토큰에 대한 가중치 분포의 합 = 1
(b) **1이 아님**. 열 합은 정규화 대상이 아니므로 임의의 값. 어떤 토큰이 다른 모든 토큰에 의해 자주 참조되면 큰 값이 됨.
(c) 토큰 3은 거의 **토큰 5에만 집중하고 있음** — 토큰 5가 토큰 3을 이해하는 데 핵심적인 역할 (예: 주어-동사 관계 등)


In [14]:
# ──────────────────────────────────────────────────────────────
# 특정 토큰이 집중하는 상위 토큰 분석
#
# "이 문장에서 '재밌어요'는 어떤 단어들에 집중하는가?"
#
# 모든 헤드의 어텐션을 평균 내어 사용 (12 헤드 → 평균 → 1 행렬)
# ──────────────────────────────────────────────────────────────

def top_attention_tokens(sentence: str, query_word: str,
                          layer: int = 11, top_k: int = 5):
    """
    query_word 토큰이 가장 집중하는 상위 K개 토큰과 가중치를 반환합니다.
    """
    inputs = tokenizer(
        sentence, return_tensors="pt",
        max_length=64, truncation=True
    ).to(DEVICE)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        out = model_bert(**inputs, output_attentions=True)

    # 지정 레이어의 모든 헤드 평균: [num_heads, seq, seq] → [seq, seq]
    # mean(dim=0)으로 첫 번째 축(헤드)을 따라 평균
    attn_avg = out.attentions[layer][0].mean(dim=0).cpu()

    # query_word에 해당하는 토큰 위치 찾기
    query_pos = None
    for i, tok in enumerate(tokens):
        if query_word in tok:
            query_pos = i
            break

    if query_pos is None:
        print(f"  '{query_word}'를 문장에서 찾을 수 없습니다.")
        return

    # 상위 K개 추출
    query_attn = attn_avg[query_pos]
    top_vals, top_idx = torch.topk(query_attn, min(top_k, len(tokens)))

    print(f"  '{query_word}' 토큰(위치 {query_pos})이 집중하는 상위 토큰 [레이어 {layer+1}]:")
    for val, idx in zip(top_vals, top_idx):
        bar = "█" * int(val.item() * 50)
        print(f"    → '{tokens[idx.item()]:10}' (위치 {idx.item():2d})  {val.item():.4f}  {bar}")


# 분석 예시
examples = [
    ("이 영화는 정말 재밌어요",         "재밌어요"),
    ("어제 카페 갔었어 거기 사람 많더라", "사람"),
    ("딥러닝 모델을 학습시킵니다",        "학습"),
]

print("=" * 65)
print("  토큰별 어텐션 집중 패턴 분석 (레이어 12 기준)")
print("=" * 65)
for sent, query in examples:
    print(f"\n문장: {sent}")
    top_attention_tokens(sent, query, layer=11)


  토큰별 어텐션 집중 패턴 분석 (레이어 12 기준)

문장: 이 영화는 정말 재밌어요
  '재밌어요'를 문장에서 찾을 수 없습니다.

문장: 어제 카페 갔었어 거기 사람 많더라
  '사람' 토큰(위치 6)이 집중하는 상위 토큰 [레이어 12]:
    → '[SEP]     ' (위치  8)  0.7660  ██████████████████████████████████████
    → '사람        ' (위치  6)  0.1139  █████
    → '많더라       ' (위치  7)  0.0360  █
    → '카페        ' (위치  2)  0.0255  █
    → '거기        ' (위치  5)  0.0203  █

문장: 딥러닝 모델을 학습시킵니다
  '학습' 토큰(위치 6)이 집중하는 상위 토큰 [레이어 12]:
    → '[SEP]     ' (위치 10)  0.5364  ██████████████████████████
    → '##시       ' (위치  7)  0.0887  ████
    → '##을       ' (위치  5)  0.0798  ███
    → '학습        ' (위치  6)  0.0703  ███
    → '모델        ' (위치  4)  0.0692  ███


### 🧮 Q5-6. 헤드 평균 계산

> `out.attentions[11][0]`의 shape이 `[12, 10, 10]`이라고 가정 (12 헤드, 10 토큰).
>
> (a) `out.attentions[11][0].mean(dim=0)`의 결과 shape은? `[__, __]`
>
> (b) `mean(dim=1)`로 잘못 계산하면 shape은? `[__, __]` — 무엇을 평균낸 것인가?
>
> (c) 12개 헤드를 모두 평균내는 이유는 무엇인가? (왜 헤드별로 보지 않고?)

**🔑 답:**
(a) **`[10, 10]`** — dim=0 (헤드 축)을 평균내어 제거. 12개 헤드 어텐션의 평균 행렬.
(b) **`[12, 10]`** — dim=1 (Query 축)을 평균내어 제거. 의미 없음 — Query별 분포가 합쳐져 해석 불가.
(c) 각 헤드는 다른 패턴을 학습하므로 **종합적인 뷰**가 필요할 때 평균을 사용. 단, 특정 헤드의 특화 패턴(예: 주-목 관계만 보는 헤드)을 분석하려면 헤드를 분리해서 봐야 함.


### 🐛 Q5-7. 오류 수정 — 어텐션 추출 (3개)

> ```python
> out = model_bert(**inputs)                            # 오류 1
> attn = out.attention[11][:, 0]                        # 오류 2
> attn_avg = attn.mean(dim=2)                           # 오류 3
> ```

**🔑 답:**
1. `output_attentions=True` 누락 → **`model_bert(**inputs, output_attentions=True)`** 로 호출
2. `attention` (단수) → **`attentions`** (복수). `[:, 0]`은 의미 모호 → **`[11][0, head]`** 같이 명확한 인덱싱 필요
3. `dim=2`로 평균을 내면 Key 축이 사라짐 → 헤드를 평균내려면 **`mean(dim=0)`** (헤드 축이 0번에 위치)


---
## 6단계. 문맥 의존적 임베딩 분석

### 🔬 왜 BERT 임베딩이 강력한가?

기존 Word2Vec 같은 정적 임베딩은 **단어마다 고정된 하나의 벡터**를 사용합니다.
BERT 임베딩은 **문맥에 따라 같은 단어도 다른 벡터**로 표현됩니다.

```
"사과를 먹었다"    → '사과' 벡터 ← 과일 의미
"사과드립니다"     → '사과' 벡터 ← 사죄 의미

Word2Vec: 두 문장의 '사과'가 완전히 동일한 벡터
BERT:     두 문장의 '사과'가 완전히 다른 벡터 ← 문맥 이해!
```

### 🧮 동작 원리 (Self-Attention 관점)

같은 토큰 '사과'라도 입력 문장 전체가 다르면:
1. Query/Key/Value 계산은 같은 가중치를 사용하지만,
2. **주변 토큰의 K, V가 완전히 다르므로**
3. Self-Attention의 가중합 결과(Z)가 완전히 달라짐

→ Word2Vec과 달리 BERT는 **단어 임베딩 = 함수(단어, 문맥 전체)**


### ❓ Q6-1. Word2Vec vs BERT 임베딩

> (a) Word2Vec에서 '배(boat)'와 '배(stomach)'는 어떤 벡터를 가지는가?
>
> (b) BERT에서 같은 두 '배'는 어떤 벡터를 가지는가?
>
> (c) 이러한 차이로 인해 BERT가 Word2Vec 대비 잘 처리하는 NLP 태스크 한 가지를 들어보세요.

**🔑 답:**
(a) **완전히 동일한 벡터** — Word2Vec은 어휘별 1개 벡터를 가지므로 동음이의어 구분 불가
(b) **완전히 다른 벡터** — Self-Attention이 주변 문맥('항구', '먹었다' 등)에 따라 다른 가중합을 계산
(c) **동음이의어 처리, 기계 번역, 질의응답(QA), 감성 분석** 등 — 같은 단어의 의미를 문맥으로 구분해야 하는 모든 태스크


### 📝 Q6-2. 빈칸 채우기 — 단어 임베딩 추출

> **다음 코드 셀의 `______` 빈칸을 채우세요.**
> 한 문장에서 특정 단어를 찾아 그 위치의 BERT 임베딩(서브워드 평균)을 가져오는 함수입니다.


In [15]:
# 🔑 Q6-2 (교수용 정답): 단어 임베딩 추출 함수
def get_word_embedding(sentence: str, target_word: str) -> torch.Tensor:
    """문장 안의 특정 단어 임베딩(서브워드 평균)을 반환"""
    # 1) 토크나이즈
    inputs = tokenizer(sentence, return_tensors="pt").to(DEVICE)   # 🔑 (a) pt
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # 2) target_word를 토크나이즈해서 서브워드 시퀀스 얻기
    target_tokens = tokenizer.tokenize(target_word)    # 🔑 (b) target_word

    # 3) 문장 토큰 리스트에서 target_tokens가 나타나는 위치 찾기
    word_indices = []
    for i in range(len(tokens) - len(target_tokens) + 1):
        if tokens[i : i + len(target_tokens)] == target_tokens:
            word_indices = list(range(i, i + len(target_tokens)))
            break

    if not word_indices:
        raise ValueError(f"'{target_word}' 토큰을 문장에서 찾지 못함. tokens={tokens}")

    # 4) 모델 forward
    with torch.no_grad():                              # 🔑 (c) no_grad
        outputs = model_bert(**inputs)
    last_hidden = outputs.last_hidden_state            # 🔑 (d) last_hidden_state

    # 5) 서브워드 위치들의 임베딩을 평균
    #     last_hidden[0, word_indices] shape: [num_subword, 768]
    #     dim=0으로 평균 → [768]
    word_emb = last_hidden[0, word_indices].mean(dim=0)   # 🔑 (e) 0 (서브워드 축으로 평균)
    return word_emb       # [768]


# 동작 확인
emb = get_word_embedding("이 영화는 재미있었어요", "영화")
print(f"✅ '영화' 임베딩 shape: {tuple(emb.shape)}  (예상: (768,))")


✅ '영화' 임베딩 shape: (768,)  (예상: (768,))


### ❓ Q6-3. 동음이의어 임베딩 해석

> (a) "배가 고파서" vs "큰 배가 들어왔다"의 '배' 벡터 유사도가 0.5 정도라면 무엇을 의미하는가?
>
> (b) "배 한 개를 깎아서" (과일) vs "배가 고파서" (신체) — 둘 다 음식과 관련. 어느 쪽 유사도가 더 높을 것으로 예상되는가? 왜?
>
> (c) Word2Vec으로 같은 실험을 하면 모든 유사도는 어떤 값이 되는가?

**🔑 답:**
(a) BERT가 두 '배'를 **상당히 다른 의미**로 인식하고 있다는 증거. 동일 단어임에도 문맥에 따라 벡터가 크게 달라짐.
(b) 둘 다 음식·식사 관련이므로 **다른 의미('선박')보다는 유사도가 높을 가능성**이 있음. 단, 문법 역할(주어 vs 목적어)에 따라 달라질 수 있음.
(c) **모두 1.0** — Word2Vec은 단어당 1개의 정적 벡터를 가지므로 같은 단어는 항상 유사도 1


### 🧮 Q6-4. 서브워드 평균 계산

> "사과를"이 WordPiece에 의해 ["사", "##과", "##를"]로 토큰화되었다.
> 각 서브워드의 BERT 출력 벡터가 다음과 같다.
>
> | 서브워드 | 벡터 (간략화, 3차원) |
> |---------|---------------------|
> | "사"    | [0.6, 0.2, 0.4]    |
> | "##과"  | [0.0, 0.4, 0.2]    |
> | "##를"  | [0.3, 0.0, 0.0]    |
>
> (a) "사과" 단어의 임베딩 (서브워드 "사" + "##과"의 평균) = ______
>
> (b) 만약 "##를"까지 포함해서 평균을 낸다면? = ______
>
> (c) 단어 임베딩에서 "##를"을 제외해야 하는 이유는?

**🔑 답:**
(a) `[(0.6+0.0)/2, (0.2+0.4)/2, (0.4+0.2)/2] = [0.30, 0.30, 0.30]`
(b) `[(0.6+0.0+0.3)/3, (0.2+0.4+0.0)/3, (0.4+0.2+0.0)/3] = [0.30, 0.20, 0.20]`
(c) "##를"은 **조사(grammatical particle)** 로 단어 의미와 무관. 단어의 의미 임베딩에는 어휘 내용만 포함하는 것이 깔끔.


In [16]:
# ──────────────────────────────────────────────────────────────
# 레이어별 임베딩 변화 추적
#
# BERT는 12개 레이어를 통과하면서 임베딩이 점차 정제됩니다.
# 초기 레이어: 단어 표면 형태 중심 (구문, 형태소)
# 후기 레이어: 문맥적 의미 중심 (의미, 화용)
#
# output_hidden_states=True 옵션:
#   outputs.hidden_states: 13개 텐서의 튜플
#     [0]:    임베딩 레이어 출력 (트랜스포머 블록 통과 전)
#     [1~12]: 각 트랜스포머 블록의 출력
# ──────────────────────────────────────────────────────────────

def get_layer_embeddings(sentence: str, word: str):
    """모든 레이어에서 특정 단어의 임베딩을 추출합니다."""
    inputs = tokenizer(
        sentence, return_tensors="pt",
        max_length=64, truncation=True
    ).to(DEVICE)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    word_positions = [i for i, t in enumerate(tokens)
                      if word in t or t.replace("##", "") in word]
    if not word_positions:
        return None, None

    with torch.no_grad():
        out = model_bert(**inputs, output_hidden_states=True)

    # 레이어별(0=임베딩, 1~12=트랜스포머) 벡터
    layer_vecs = [hidden[0, word_positions].mean(dim=0).cpu()
                  for hidden in out.hidden_states]
    return layer_vecs, tokens


# 두 문장에서 같은 단어의 레이어별 유사도 변화
s1 = "배가 고파서 밥을 먹었다"
s2 = "항구에 큰 배가 들어왔다"
word = "배"

v1_layers, _ = get_layer_embeddings(s1, word)
v2_layers, _ = get_layer_embeddings(s2, word)

if v1_layers and v2_layers:
    print("=" * 65)
    print(f"  레이어별 '{word}' 임베딩 유사도 변화")
    print(f"  문장1: {s1}")
    print(f"  문장2: {s2}")
    print("=" * 65)
    print(f"  {'레이어':>10}   {'코사인 유사도':>14}   시각화")
    print("  " + "─" * 60)

    for i, (v1, v2) in enumerate(zip(v1_layers, v2_layers)):
        sim = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
        bar_len = int(sim * 35)
        bar = "█" * bar_len + "░" * (35 - bar_len)
        label = "(임베딩)" if i == 0 else f"(레이어{i:2d})"
        print(f"  {label:>10}   {sim:14.4f}   {bar}")

    print()
    print("  📌 레이어가 깊어질수록 유사도가 낮아진다면,")
    print("     모델이 점점 더 '배'의 문맥적 의미를 구분하고 있다는 증거입니다.")


  레이어별 '배' 임베딩 유사도 변화
  문장1: 배가 고파서 밥을 먹었다
  문장2: 항구에 큰 배가 들어왔다
         레이어          코사인 유사도   시각화
  ────────────────────────────────────────────────────────────
       (임베딩)           0.9298   ████████████████████████████████░░░
     (레이어 1)           0.8168   ████████████████████████████░░░░░░░
     (레이어 2)           0.7337   █████████████████████████░░░░░░░░░░
     (레이어 3)           0.7215   █████████████████████████░░░░░░░░░░
     (레이어 4)           0.7014   ████████████████████████░░░░░░░░░░░
     (레이어 5)           0.6703   ███████████████████████░░░░░░░░░░░░
     (레이어 6)           0.6615   ███████████████████████░░░░░░░░░░░░
     (레이어 7)           0.6291   ██████████████████████░░░░░░░░░░░░░
     (레이어 8)           0.5672   ███████████████████░░░░░░░░░░░░░░░░
     (레이어 9)           0.4975   █████████████████░░░░░░░░░░░░░░░░░░
     (레이어10)           0.4578   ████████████████░░░░░░░░░░░░░░░░░░░
     (레이어11)           0.4538   ███████████████░░░░░░░░░░░░░░░░░░░░
     (레이어12)         

### ❓ Q6-5. 레이어별 임베딩 변화

> (a) 일반적으로 초기 레이어(0~3)에서 같은 단어의 두 문맥 임베딩 유사도는 어떻게 나타날까? (높음/낮음)
>
> (b) 후기 레이어(9~12)에서는?
>
> (c) 이러한 변화가 의미하는 것은? — BERT가 어떤 정보를 어떻게 처리하는지의 관점에서

**🔑 답:**
(a) **높음** — 초기 레이어는 토큰의 표면 형태(같은 글자 '배')에 가깝게 표현되므로 두 문맥의 임베딩이 비슷
(b) **낮음** — 후기 레이어는 문맥 정보가 누적되어 의미적으로 다른 두 '배'가 다른 벡터로 분화
(c) BERT는 **하위 → 상위 레이어로 갈수록 점점 더 추상적·문맥적·의미적인 표현**을 학습. 이것이 BERT를 다양한 다운스트림 태스크에 활용 가능하게 하는 핵심 메커니즘.


### 🔢 Q6-6. `hidden_states` 길이

> `out.hidden_states`의 길이는 정확히 ____ 개이며, 그 이유는 ____ 이다.
>
> (a) 길이: ______
>
> (b) `out.hidden_states[0]`이 의미하는 것: ______
>
> (c) `out.hidden_states[12]`와 `out.last_hidden_state`의 관계: ______

**🔑 답:**
(a) **13개** — 임베딩 레이어 출력(0번) + 트랜스포머 블록 12개 출력(1~12번)
(b) **임베딩 레이어 출력** — 토큰 임베딩 + 위치 임베딩 + 세그먼트 임베딩 합산 후 LayerNorm 통과 결과 (트랜스포머 블록 통과 전)
(c) **완전히 동일한 텐서** — `out.hidden_states[12] == out.last_hidden_state` (같은 데이터의 두 가지 접근 방식)


---
## 📝 강의 요약

### 핵심 개념 정리

| 항목 | 내용 |
|------|------|
| **언어 모델** | 단어 시퀀스에 확률을 부여. $P(w_1, ..., w_n) = \prod_i P(w_i \mid w_{<i})$ |
| **GPT (순방향 LM)** | 앞 단어만 보고 다음 단어 예측. 텍스트 생성에 적합. |
| **BERT (MLM)** | 앞뒤 문맥 모두 보고 빈칸 예측. 양방향. 문장 이해에 적합. |
| **트랜스포머** | 인코더 블록 12개. Self-Attention + FFN. 12헤드 병렬 처리. |
| **Self-Attention** | $\text{softmax}(QK^T/\sqrt{d_k}) V$ — 모든 토큰 쌍의 관계를 동시에 계산. |
| **Q/K/V** | Query=질의, Key=색인, Value=실제 내용. 헤드별로 다른 패턴 학습. |
| **`[CLS]` 벡터** | BERT 마지막 레이어의 [CLS] 출력 (또는 pooler_output). 문장 전체 의미 집약. |
| **`last_hidden_state`** | `[batch, seq_len, 768]` — 모든 토큰의 마지막 레이어 벡터 |
| **`pooler_output`** | `[batch, 768]` — [CLS] 위치 + Linear + tanh |
| **문맥 의존 임베딩** | 같은 단어도 문맥에 따라 다른 벡터. BERT의 핵심 강점. |
| **Fine-tuning** | 사전학습 모델 위에 작은 분류기 추가 → 다운스트림 태스크 수행. |

### 핵심 함수/메서드 정리

| 함수/메서드 | 역할 |
|------------|------|
| `tokenizer(text, return_tensors="pt")` | 텍스트 → 토큰 ID 텐서 |
| `tokenizer.convert_ids_to_tokens(ids)` | 토큰 ID → 토큰 문자열 |
| `tokenizer.mask_token_id` | [MASK] 토큰의 ID (KcBERT는 4) |
| `BertModel.from_pretrained(name)` | 사전학습 가중치 로드 |
| `model.eval()` | 추론 모드 (dropout off) |
| `with torch.no_grad():` | gradient 계산 끄기 |
| `model_bert(**inputs, output_attentions=True)` | 어텐션 가중치 함께 반환 |
| `outputs.last_hidden_state` | `[batch, seq_len, hidden]` |
| `outputs.pooler_output` | `[batch, hidden]` |
| `outputs.hidden_states` | tuple of 13 (임베딩 + 12 layers) |
| `outputs.attentions` | tuple of 12 `[batch, heads, seq, seq]` |
| `F.softmax(x, dim=-1)` | 확률화 (마지막 축 따라) |
| `F.cosine_similarity(v1, v2)` | 코사인 유사도 |
| `torch.topk(x, k)` | 상위 K개 값과 인덱스 |


---
## 📋 종합 퀴즈 (중간고사 대비) — 정답 포함

> 24개 퀴즈 모두 정답이 함께 제공됩니다.
> 학생들이 막히는 부분에서 보충 설명을 해 주세요.


### Quiz 1. 언어 모델의 정의

> 언어 모델이란 단어 시퀀스에 ______ 을 부여하는 모델이며,
> 결합 확률 $P(w_1, w_2, ..., w_n)$은 조건부 확률의 ______ 로 분해된다.

**🔑 답:** 확률(probability) / 연쇄(chain rule, 곱)


### Quiz 2. 학습 방식 분류

> | 모델 | 학습 방식 | 방향 |
> |------|---------|------|
> | GPT  | ______ | ______ |
> | BERT | ______ | ______ |
> | ELMo | ______ + ______ | 양방향 (단방향 결합) |
> | Word2Vec | ______ | 중심→주변 |

**🔑 답:**
- GPT: **순방향 LM** / **왼쪽→오른쪽**
- BERT: **마스크 LM (MLM)** / **양방향**
- ELMo: **순방향 LM + 역방향 LM**
- Word2Vec: **스킵-그램 (Skip-gram)**


### Quiz 3. KcBERT 하이퍼파라미터

> KcBERT-base의 핵심 설정값을 채우세요.
>
> - vocab_size = ______
> - hidden_size = ______
> - num_hidden_layers = ______
> - num_attention_heads = ______
> - intermediate_size = ______
> - 헤드당 차원 = ______ ÷ ______ = ______
> - 전체 파라미터 수 ≈ ______ M

**🔑 답:**
vocab_size = **30000**, hidden_size = **768**, num_hidden_layers = **12**,
num_attention_heads = **12**, intermediate_size = **3072**,
헤드당 차원 = **768 ÷ 12 = 64**, 전체 파라미터 ≈ **110M (1.1억)**


### Quiz 4. BERT 입력 형식

> `tokenizer("안녕")`의 결과 input_ids는 어떤 형식이 되는가?
>
> [______, ______, ______]   ← 인덱스 값으로 답하세요 (KcBERT 기준)

**🔑 답:** [**2**(=[CLS]), **'안녕' 토큰 ID**, **3**(=[SEP])]
KcBERT 특수토큰: [PAD]=0, [UNK]=1, [CLS]=2, [SEP]=3, [MASK]=4


### Quiz 5. Self-Attention 수식

> Self-Attention의 출력 계산 공식 (Scaled Dot-Product Attention):
>
> $$\text{Attention}(Q, K, V) = \text{______}\left(\frac{______}{\sqrt{______}}\right) \cdot ______$$

**🔑 답:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) \cdot V$$


### Quiz 6. Q/K/V 역할

> 도서관 검색 비유로 Q/K/V를 한 단어씩 답하세요.
>
> - Q (Query): ______
> - K (Key): ______
> - V (Value): ______

**🔑 답:**
- Q (Query): **검색 질의문/검색어**
- K (Key): **책의 색인/제목** (질의와 매칭되는 부분)
- V (Value): **책의 실제 내용** (검색 결과로 가져올 정보)


### Quiz 7. 가중합 계산

> '거기' 토큰에 대해 다른 토큰들의 어텐션 가중치가 다음과 같다.
>
> α_어제 = 0.05, α_카페 = 0.40, α_갔었어 = 0.30, α_거기 = 0.10, α_사람 = 0.10, α_많더라 = 0.05
>
> (a) 가중치 합 = ______
>
> (b) V_카페 = [1.0, 0.0], V_갔었어 = [0.0, 1.0]이고 다른 V는 모두 [0,0]일 때, Z_거기 = ______

**🔑 답:**
(a) 0.05 + 0.40 + 0.30 + 0.10 + 0.10 + 0.05 = **1.0**
(b) Z_거기 = 0.40 × [1.0, 0.0] + 0.30 × [0.0, 1.0] + 0 = **[0.40, 0.30]**


### Quiz 8. Shape 계산 (BERT 출력)

> KcBERT에 batch=4, max_length=32 문장을 입력하면 다음 shape은?
>
> (a) `outputs.last_hidden_state.shape` = `[__, __, __]`
>
> (b) `outputs.pooler_output.shape` = `[__, __]`
>
> (c) `outputs.attentions[0].shape` = `[__, __, __, __]`
>
> (d) `len(outputs.hidden_states)` = ______

**🔑 답:**
(a) `[4, 32, 768]`
(b) `[4, 768]`
(c) `[4, 12, 32, 32]` — [batch, num_heads, seq_len, seq_len]
(d) **13** (임베딩 + 12 트랜스포머 블록)


### Quiz 9. 트랜스포머 블록 구성요소

> 트랜스포머 블록을 구성하는 4가지 핵심 모듈을 순서대로 적으세요.
>
> 1. ______
> 2. ______
> 3. ______
> 4. ______

**🔑 답:**
1. **Multi-Head Self-Attention** (멀티 헤드 셀프 어텐션)
2. **Add & Norm** (잔차 연결 + 레이어 정규화)
3. **Feed-Forward Network** (FFN, 위치별 완전연결망 768→3072→768)
4. **Add & Norm** (잔차 연결 + 레이어 정규화)


### Quiz 10. 빈칸 채우기 — 패턴 코드

> ```python
> from transformers import ______, ______               # (a)(b)
> tokenizer = ______.from_pretrained("beomi/kcbert-base", do_lower_case=______)  # (c)(d)
> model = ______.from_pretrained("beomi/kcbert-base")    # (e)
> model.______()                                          # (f)
> model.______(DEVICE)                                    # (g)
>
> inputs = tokenizer("문장", return_tensors="____").to(DEVICE)  # (h)
> with torch.________():                                          # (i)
>     out = model(**inputs)
> cls = out.________                                              # (j) [CLS] pooler 벡터
> ```

**🔑 답:**
(a) `BertTokenizer`  (b) `BertModel`  (c) `BertTokenizer`  (d) `False`
(e) `BertModel`  (f) `eval`  (g) `to`  (h) `pt`  (i) `no_grad`  (j) `pooler_output`


### Quiz 11. 오류 찾기 — Self-Attention 추출

> ```python
> out = model_bert(**inputs)                       # 오류 ?
> attn = out.attention[11].mean(dim=0)             # 오류 ?
> ```
>
> 두 줄에서 오류 2개를 찾고 수정하세요.

**🔑 답:**
1. `output_attentions=True` 누락 → **`model_bert(**inputs, output_attentions=True)`**
2. `attention` (단수) → **`attentions`** (복수). 또한 `attentions[11]` shape은 `[batch, heads, seq, seq]`이므로 헤드 평균을 내려면 **`out.attentions[11][0].mean(dim=0)`** (배치 0 선택 후 헤드 차원 평균)


### Quiz 12. 코사인 유사도 직접 계산

> 두 문장 임베딩이 다음과 같다 (간략화한 4차원 벡터):
> v1 = [1, 2, 2, 1], v2 = [2, 1, 1, 2]
>
> (a) v1 · v2 = ______
>
> (b) |v1| = ______ (= |v2|)
>
> (c) cos(v1, v2) = ______ / ______ = ______

**🔑 답:**
(a) v1·v2 = 1×2 + 2×1 + 2×1 + 1×2 = 2+2+2+2 = **8**
(b) |v1| = √(1+4+4+1) = √10 ≈ **3.162**, |v2| = √10 ≈ **3.162**
(c) cos = 8 / (√10 × √10) = 8 / 10 = **0.8**


### Quiz 13. 문맥 의존 임베딩

> "사과를 먹었다"의 '사과' 벡터와 "사과드립니다"의 '사과' 벡터의 코사인 유사도를 계산했다.
>
> (a) Word2Vec에서의 결과: ______
>
> (b) BERT에서의 예상 결과: ______ (높음/낮음 + 그 이유)

**🔑 답:**
(a) **1.0** — Word2Vec은 단어당 1개의 정적 벡터를 가지므로 항상 동일
(b) **낮음 (보통 0.4~0.7)** — Self-Attention이 주변 문맥('먹었다' vs '드립니다')에 따라 다른 가중합을 계산하므로 의미적으로 다른 두 '사과'를 다른 벡터로 표현


### Quiz 14. `attn_implementation`

> `BertModel.from_pretrained("...", attn_implementation="____")` 빈칸에 들어갈 수 있는 두 옵션은?
>
> (a) ______
>
> (b) ______
>
> (c) Self-Attention 가중치 시각화를 위해서는 어느 쪽? ______

**🔑 답:**
(a) `eager` — 표준 구현, attention weight 추출 가능
(b) `sdpa` — PyTorch 2.0+의 fused 커널, 빠르지만 weight 추출 불가
(c) **`eager`** — `output_attentions=True`가 작동하려면 `eager` 필수


### Quiz 15. `hidden_states` 길이

> `out.hidden_states`의 길이는 ______ 이다.
>
> 그 구성:
> - 인덱스 0: ______ 의 출력
> - 인덱스 1~12: ______ 의 출력
>
> `out.hidden_states[12]`는 `out.____________` 와 동일하다.

**🔑 답:**
길이 = **13**
- 인덱스 0: **임베딩 레이어** (토큰 + 위치 + 세그먼트 임베딩 합산 후 LayerNorm)
- 인덱스 1~12: **각 트랜스포머 블록** (12개)
- `out.hidden_states[12]` == `out.last_hidden_state` ✓


### Quiz 16. MLM Top-K 추출 빈칸

> ```python
> inputs = tokenizer("[MASK] 좋아요", return_tensors="pt").to(DEVICE)
> mask_idx = (inputs['input_ids'][0] == tokenizer.________).nonzero(as_tuple=True)[0]  # (a)
> with torch.________():                                                                  # (b)
>     logits = mlm_model(**inputs).________                                                 # (c)
> probs = F.________(logits[0, mask_idx[0]], dim=____)                                    # (d)(e)
> top_vals, top_ids = torch.________(probs, 10)                                           # (f)
> tokens = [tokenizer.________([i.item()])[0] for i in top_ids]                            # (g)
> ```

**🔑 답:**
(a) `mask_token_id`  (b) `no_grad`  (c) `logits`
(d) `softmax`  (e) `-1`  (f) `topk`  (g) `convert_ids_to_tokens`


### Quiz 17. 파라미터 수 계산

> KcBERT-base의 토큰 임베딩 행렬 shape이 [vocab_size=30000, hidden=768]일 때, 토큰 임베딩 파라미터 수는?
>
> (a) ______ × ______ = ______ 개 (대략 ______ M)
>
> (b) Self-Attention 한 블록의 Q, K, V 가중치 행렬 (각 [768, 768])의 총 파라미터 수: ______
>
> (c) FFN 한 블록의 가중치 행렬 ([768, 3072] + [3072, 768])의 총 파라미터 수: ______

**🔑 답:**
(a) 30000 × 768 = **23,040,000** 개 (~**23M**)
(b) 768 × 768 × 3 = **1,769,472** 개 (~1.77M)
(c) 768 × 3072 + 3072 × 768 = 2,359,296 × 2 = **4,718,592** 개 (~4.72M)


### Quiz 18. 어텐션 행렬의 행 합

> Self-Attention 가중치 행렬 attn (shape [seq_len, seq_len])에 대해:
>
> (a) `attn[i].sum()` (한 행의 합) = ______
>
> (b) 그 이유는 무엇인가? 어떤 함수가 적용되었기 때문인가?
>
> (c) `attn[:, j].sum()` (한 열의 합) = ______ (구체적인 값 또는 "임의값")

**🔑 답:**
(a) 항상 **1.0** (정확히 1)
(b) **softmax 함수**가 행(query) 단위로 적용되었기 때문. softmax는 합이 1인 확률 분포를 만듦.
(c) **임의값** (정해진 합 없음) — 열 방향은 정규화 대상이 아님. 어떤 토큰이 자주 참조되면 큰 값.


### Quiz 19. 트랜스포머 인코더 vs 디코더

> 강의노트의 트랜스포머 구조에서:
>
> (a) **인코더**의 입력은? ______
>
> (b) **디코더**의 입력은? ______
>
> (c) 디코더에는 인코더에 없는 ______ 와 ______ 두 모듈이 추가된다.

**🔑 답:**
(a) **소스 시퀀스 전체** (예: 한국어 문장 "어제 카페 갔었어 거기 사람 많더라")
(b) **타깃 시퀀스 일부** (지금까지 생성된 부분, 예: `<s> I went`)
(c) **마스크 멀티 헤드 어텐션** (Masked Multi-Head Attention) + **인코더-디코더 어텐션** (Encoder-Decoder Multi-Head Attention)


### Quiz 20. 학습 vs 인퍼런스 차이

> 트랜스포머 학습 시 디코더의 입력과 인퍼런스 시 디코더의 입력 차이를 설명하세요.
>
> (a) 학습 시: 디코더 입력 = ______
>
> (b) 인퍼런스 시: 디코더 입력 = ______

**🔑 답:**
(a) **정답 타깃 시퀀스 (그 시점까지의 ground truth)** — Teacher Forcing 방식. 예: `<s> I went`을 입력하고 `to`를 예측하도록 학습.
(b) **직전까지 디코더가 생성한 출력 시퀀스** — Auto-regressive 방식. 첫 단계는 `<s>`만 입력 → 출력 `I`, 다음 단계 `<s> I` 입력 → 출력 `went`, 이렇게 한 토큰씩 누적.


### Quiz 21. CNN/RNN과 Self-Attention 비교

> 강의노트에서 설명한 Self-Attention의 장점을 두 모델과 비교하여 답하세요.
>
> (a) CNN의 한계: ______ 를 넘어서는 문맥은 읽기 어려움
>
> (b) RNN의 한계: 시퀀스 길이가 길어질수록 ______ 에 문제
>
> (c) Self-Attention의 강점: ______

**🔑 답:**
(a) **합성곱 필터 크기 (kernel size)** 를 넘어서는 장거리 문맥은 잡기 어려움
(b) **정보 압축** (information bottleneck) — 오래전 단어는 잊거나 특정 단어가 과도하게 반영되어 정보 왜곡
(c) **모든 토큰 쌍의 관계를 한 번에 직접 계산**하므로 시퀀스 길이가 길어져도 정보 손실 없이 처리 가능. + 병렬화 용이 (RNN과 달리 순차 처리 불필요)


### Quiz 22. 종합 — BERT 추론 코드 빈칸

> 다음은 KcBERT로 두 문장의 의미적 유사도를 계산하는 전체 코드입니다.
> 빈칸을 모두 채우세요.
>
> ```python
> from transformers import ______, ______               # (a)(b)
> import torch
> import torch.nn.functional as F
>
> tokenizer = ______.from_pretrained("beomi/kcbert-base", do_lower_case=False)  # (c)
> model     = ______.from_pretrained("beomi/kcbert-base")                         # (d)
> model.______()                                                                   # (e)
>
> def get_emb(sent):
>     inputs = tokenizer(sent, return_tensors="____", padding=True, truncation=True)  # (f)
>     with torch.________():                                                              # (g)
>         out = model(**inputs)
>     return out.________[0]                                                              # (h)
>
> v1 = get_emb("이 영화 재밌어요")
> v2 = get_emb("이 영화 별로예요")
> sim = F.________(v1.________(0), v2.________(0)).item()                                # (i)(j)(k)
> print(sim)
> ```

**🔑 답:**
(a) `BertTokenizer`  (b) `BertModel`  (c) `BertTokenizer`  (d) `BertModel`  (e) `eval`
(f) `pt`  (g) `no_grad`  (h) `pooler_output`
(i) `cosine_similarity`  (j) `unsqueeze`  (k) `unsqueeze`


### Quiz 23. 종합 서술 (선택)

> "BERT는 왜 양방향 학습이 가능한가?"를 GPT와 비교하여 3~5문장으로 서술하세요.

**🔑 답 (예시):**
GPT는 텍스트 생성 모델로 "다음 단어 예측" 태스크로 학습되며, 미래 정보를 보면 cheating이 되므로 단방향(왼쪽→오른쪽) 마스킹된 어텐션을 사용합니다. 반면 BERT는 "마스크된 단어 복원(MLM)" 태스크로 학습되며, 입력의 일부 단어만 [MASK]로 가리고 나머지 앞뒤 모든 단어를 컨텍스트로 활용합니다. 따라서 어텐션에 단방향 마스크가 필요하지 않아 모든 위치의 토큰이 서로를 자유롭게 참조할 수 있습니다(양방향). 이로 인해 BERT는 문장 이해(분류, NER, QA 등)에 강하고, GPT는 자연스러운 텍스트 생성에 강한 특성을 가집니다.


### Quiz 24. 종합 — 실전 응용 (선택)

> KcBERT를 사용하여 영화 리뷰 감성 분석(긍정/부정 분류)을 수행하려고 합니다.
> 다음 중 올바른 접근은? (정답 여러 개 가능)
>
> (a) `pooler_output`을 추출해 새로운 Linear 분류기를 추가
>
> (b) `last_hidden_state[:, 0, :]`을 추출해 새로운 Linear 분류기를 추가
>
> (c) `last_hidden_state.mean(dim=1)`을 추출해 새로운 Linear 분류기를 추가
>
> (d) MLM 헤드를 그대로 사용해 분류
>
> (e) 모든 파라미터를 freeze하고 분류 헤드만 학습 (linear probing)
>
> (f) 모든 파라미터를 함께 fine-tuning

**🔑 답:**
- (a) ✓ 가능 — pooler가 NSP에 학습되어 있으므로 분류에도 자주 사용
- (b) ✓ 가능 — pooler 없이 raw [CLS] 사용, 실제로 더 자주 권장됨
- (c) ✓ 가능 — mean pooling 방식, 문장 임베딩에서 흔히 사용
- (d) ✗ 잘못됨 — MLM 헤드는 vocab_size 차원으로 출력, 감성 2 클래스에 맞지 않음
- (e) ✓ 가능 — linear probing, 데이터가 적을 때 유용
- (f) ✓ 가능 — 표준 fine-tuning, 데이터가 충분할 때 가장 강력

→ 정답: **(a), (b), (c), (e), (f)**



---
### 📚 참고 자료

- [ratsgo's NLPBOOK — Pretrained LM](https://ratsgo.github.io/nlpbook/docs/language_model/semantics/)
- [ratsgo's NLPBOOK — Transformers](https://ratsgo.github.io/nlpbook/docs/language_model/transformers/)
- [ratsgo's NLPBOOK — Embedding Tutorial](https://ratsgo.github.io/nlpbook/docs/language_model/tutorial/)
- [KcBERT HuggingFace Hub](https://huggingface.co/beomi/kcbert-base)
- [BERT 논문 (Devlin et al., 2018)](https://arxiv.org/abs/1810.04805)
- [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)

---

> 💬 **수업 후 질문은 강의 직후 또는 office hour에 받겠습니다.**
> 이 노트북의 빈칸과 퀴즈를 모두 풀어보면 중간고사 코딩 문제 대비가 충분합니다.
